# Funding Rate parameter for BTC data

In [2]:
import os
from typing import List
import numpy as np
import pandas as pd
import plotly.express as px
pd.options.plotting.backend = "plotly"

import sys
sys.path.append(os.getcwd().split("scripts")[0])
sys.path.append(os.path.join(os.getcwd().split("scripts")[0], "params"))

from params import funding, caps
import pystable
from tqdm import tqdm

## Loading the Original Data

In [3]:
filename = "btc"

path_to_file = os.path.join(os.getcwd().split("scripts")[0], f"data/{filename}")

periodicity = 60. # 1 minute in seconds

short_twap = 10
long_twap = 60

sample_rate = 10  # in minutes: it samples the data set every ten minutes

In [4]:
df = pd.read_csv(path_to_file+".csv", parse_dates=["timestamp"]).set_index("timestamp")
df

,close
timestamp,
2024-01-01 00:01:00+00:00,42298.61
2024-01-01 00:02:00+00:00,42320.00
2024-01-01 00:03:00+00:00,42325.50
2024-01-01 00:04:00+00:00,42367.99
2024-01-01 00:05:00+00:00,42397.23
...,...
2024-05-08 13:06:00+00:00,62097.67
2024-05-08 13:07:00+00:00,62077.49
2024-05-08 13:08:00+00:00,62115.51


## Computing Short and Long TWAPs

In [5]:
def compute_twap(_df, _window, _rate) -> pd.DataFrame:
    return _df.rolling(_window).mean().dropna().resample(f'{_rate}min', label="right").last()

In [6]:
def concatenate_datasets(
    _df:pd.DataFrame, _column:str, _short_window:int, _long_window:int, _rate:str
) -> pd.DataFrame:

    if not _column in _df.columns:
        raise Exception(F"Column {_column} not found")

    df_short_twap = compute_twap(_df, _short_window, _rate)
    df_long_twap = compute_twap(_df, _long_window, _rate)

    first_idx = np.max([
        _df.index[0], df_short_twap.index[0], df_long_twap.index[0],
    ])

    last_idx = np.min([
        _df.index[-1], df_short_twap.index[-1], df_long_twap.index[-1],
    ])

    tmp_df = pd.DataFrame(
        _df.loc[first_idx:last_idx,_column], columns=[_column]
    )

    tmp_df["long_twap"] = df_long_twap.loc[first_idx:last_idx, :]
    tmp_df["short_twap"] = df_short_twap.loc[first_idx:last_idx, :]
    tmp_df.dropna(inplace=True)
    
    return tmp_df

In [7]:
df_twaps = concatenate_datasets(df, "close", short_twap, long_twap, sample_rate)
df_twaps

,close,long_twap,short_twap
timestamp,,,
2024-01-01 01:10:00+00:00,42436.80,42445.579667,42451.141
2024-01-01 01:20:00+00:00,42493.61,42443.685833,42467.475
2024-01-01 01:30:00+00:00,42493.16,42450.079333,42497.100
2024-01-01 01:40:00+00:00,42717.52,42485.129333,42597.026
2024-01-01 01:50:00+00:00,42698.01,42528.335833,42700.103
...,...,...,...
2024-05-08 12:30:00+00:00,62343.99,62325.059833,62326.096
2024-05-08 12:40:00+00:00,62172.15,62331.672000,62279.674
2024-05-08 12:50:00+00:00,62123.50,62302.007667,62162.789


In [8]:
df_twaps.plot()

In [9]:
def compute_log_return(_df:pd.DataFrame, columns:List[str]) -> pd.DataFrame:
    
    tmp_df = _df.copy()

    for c in columns:

        if not c in tmp_df.columns:
            raise Exception(f"Column {c} not found")

        tmp_df[f"{c}_log_return"] = np.nan
        tmp_df.loc[tmp_df.index[1]:,f"{c}_log_return"] = np.log(
            tmp_df[c].iloc[1:].to_numpy() / tmp_df[c].iloc[:-1].to_numpy()
        )

    tmp_df.dropna(inplace=True)

    return tmp_df

In [10]:
df_twaps = compute_log_return(df_twaps, columns=list(df_twaps.columns))
df_twaps

,close,long_twap,short_twap,close_log_return,long_twap_log_return,short_twap_log_return
timestamp,,,,,,
2024-01-01 01:20:00+00:00,42493.61,42443.685833,42467.475,0.001338,-0.000045,0.000385
2024-01-01 01:30:00+00:00,42493.16,42450.079333,42497.100,-0.000011,0.000151,0.000697
2024-01-01 01:40:00+00:00,42717.52,42485.129333,42597.026,0.005266,0.000825,0.002349
2024-01-01 01:50:00+00:00,42698.01,42528.335833,42700.103,-0.000457,0.001016,0.002417
2024-01-01 02:00:00+00:00,42613.56,42562.376667,42661.415,-0.001980,0.000800,-0.000906
...,...,...,...,...,...,...
2024-05-08 12:30:00+00:00,62343.99,62325.059833,62326.096,-0.000030,0.000357,-0.001028
2024-05-08 12:40:00+00:00,62172.15,62331.672000,62279.674,-0.002760,0.000106,-0.000745
2024-05-08 12:50:00+00:00,62123.50,62302.007667,62162.789,-0.000783,-0.000476,-0.001879


## Computing Current Funding Rate parameters

(Code adapted from `funding.py`)

In [11]:
def fit_rescaled_distribution(
    _df:pd.DataFrame, columns:List[str], scale_param:float, verbose:bool=False
) -> List[pystable.STABLE_DIST]:

    list_dist = list()

    for c in columns:

        _any_dst = funding.gaussian()

        pystable.fit(_any_dst, _df[c].to_numpy(), _df.index.size)

        if verbose:
            print(f"""
                original data: {c}
                alpha: {_any_dst.contents.alpha}, beta: {_any_dst.contents.beta}
                mu: {_any_dst.contents.mu_1}, sigma: {_any_dst.contents.sigma}"
                """
            )

        _any_dst = caps.rescale(_any_dst, scale_param)

        if verbose:
            print(f"""
                original data: {c}, scale: {scale_param}
                alpha: {_any_dst.contents.alpha}, beta: {_any_dst.contents.beta}
                mu: {_any_dst.contents.mu_1}, sigma: {_any_dst.contents.sigma}"
                """
            )
        
        list_dist.append(_any_dst)

    return list_dist

In [12]:
dst_close, dst_long_twap, dst_short_twap = fit_rescaled_distribution(
    df_twaps, 
    ["close_log_return", "long_twap_log_return", "short_twap_log_return"],
    scale_param=1./(sample_rate*periodicity), 
    verbose=True
)


                original data: close_log_return
                alpha: 1.4491616140753347, beta: 0.01715120236392847
                mu: 3.915354147634908e-05, sigma: 0.001130396173101184"
                

                original data: close_log_return, scale: 0.0016666666666666668
                alpha: 1.4491616140753347, beta: 0.01715120236392847
                mu: 6.52559024605818e-08, sigma: 1.7674013597384443e-05"
                

                original data: long_twap_log_return
                alpha: 1.4355786861726687, beta: 0.043264000530938616
                mu: 3.329262194135939e-05, sigma: 0.0004264525182291757"
                

                original data: long_twap_log_return, scale: 0.0016666666666666668
                alpha: 1.4355786861726687, beta: 0.043264000530938616
                mu: 5.5487703235598987e-08, sigma: 6.3685352591433554e-06"
                

                original data: short_twap_log_return
                alpha: 1.4669034993571095, 

In [13]:
def funding_rate(_any_dst:List[pystable.STABLE_DIST], flags:List[str]) -> pd.DataFrame:

    df_final_ks = pd.DataFrame(
        0., columns=["tmp"], index=[int(n/funding.NS[0]) for n in funding.NS]
    )

    for _dst, flag in zip(_any_dst, flags):

        ks = [
            funding.k(
                a=_dst.contents.alpha, b=_dst.contents.beta, 
                mu=_dst.contents.mu_1, sig=_dst.contents.sigma,
                n=_n, alphas=funding.ALPHAS
            )
            for _n in funding.NS
        ]

        df_ks = pd.DataFrame(
            data=ks,
            columns=[f"{flag}_{alpha:.3f}" for alpha in funding.ALPHAS],
            index=[int(n/funding.NS[0]) for n in funding.NS]
        )

        for c in df_ks.columns:
            df_final_ks.loc[:,c] = df_ks[c]

    df_final_ks.drop("tmp", axis=1, inplace=True)
    df_final_ks.index.name = "periods"
    df_final_ks.columns.name = "alpha"

    return df_final_ks

In [14]:
df_ks = funding_rate([dst_close, dst_long_twap, dst_short_twap], ["close", "long", "short"])
df_ks.plot(
    title="Funding rate of spot, long, short twap at 95%"
)

## Transient Funding Rate

How does the funding rate evolve with the arrival of new data?

In [15]:
df_ks_dist = pd.DataFrame(
    0.,
    columns=[f"{flag}_{alpha:.3f}" for alpha in funding.ALPHAS for flag in ["full", "partial"]],
    index=pd.date_range(
        start=df_twaps.index[0].floor("D")+pd.Timedelta(days=30), 
        end=df_twaps.index[-1].floor("D")+pd.Timedelta(days=1), 
        freq='1D'
    )
)

# from recommendation:
full_alphas = [f"full_{alpha:.3f}" for alpha in funding.ALPHAS]
partial_alphas = [f"partial_{alpha:.3f}" for alpha in funding.ALPHAS]
period = 30  # days

pbar = tqdm(total=df_ks_dist.index.size)

for idx in df_ks_dist.index:

    # --------------
    # full past data
    # --------------

    dst_close_full:List[pystable.STABLE_DIST] = fit_rescaled_distribution(
        df_twaps.loc[:idx,:], ["close_log_return"], scale_param=1./(sample_rate*periodicity), verbose=False
    )

    cur_ks_full:pd.DataFrame = funding_rate(dst_close_full, ["close"])

    df_ks_dist.loc[idx,full_alphas] = cur_ks_full.loc[period,:].to_numpy()

    # -----------------
    # partial past data - 1 month lookback window
    # -----------------

    past_idx = idx - pd.Timedelta(days=30)

    cur_dist_partial:List[pystable.STABLE_DIST] = fit_rescaled_distribution(
        df_twaps.loc[past_idx:idx,:], ["close_log_return"], scale_param=1./(sample_rate*periodicity), verbose=False
    )

    cur_ks_partial:pd.DataFrame = funding_rate(cur_dist_partial, ["close"])

    df_ks_dist.loc[idx,partial_alphas] = cur_ks_partial.loc[period,:].to_numpy()

    pbar.update(1)

pbar.close()


100%|██████████| 100/100 [00:17<00:00,  5.62it/s]


In [16]:
df_ks_dist.plot(
    title=f"Evolution of Funding Rate with {period} days anchor time, 95% confidence level"
)

## Simulation

Funding rate relies on the evaluation of VaR. How accurate is the analytical evaluation in comparison with the numerical one?